# Survey-only daily adherence models

Compares logistic regression, multilayer perceptron, random forest, support vector machine, and XGBoost using participant-level cross-validation. All preprocessing is fitted within each training fold.


In [ ]:
from pathlib import Path
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260819
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings('ignore')
DATA_ROOT = Path('TERA Analysis')
PROJECT_ROOT = Path('TERA')
RESULTS = PROJECT_ROOT / 'results'; FIGURES = PROJECT_ROOT / 'figures'
RESULTS.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)
print('Data:', DATA_ROOT)
print('Outputs:', PROJECT_ROOT)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import make_scorer, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier=None
df=pd.read_csv(DATA_ROOT/'exploratory_analysis/tera_daily.csv').dropna(subset=['USUBJID','withinrange']).reset_index(drop=True)
y=df.withinrange.astype(int); groups=df.USUBJID.astype(int)
dynamic_tokens=('t-','is_Weekend','is_adherent','Morning','Afternoon','Evening','Night','ADY','withinrange','USUBJID')
static_columns=[c for c in df.columns if not c.startswith('t-') and c not in dynamic_tokens]
X=df[static_columns]; categorical=[c for c in static_columns if X[c].dtype=='object']; numeric=[c for c in static_columns if c not in categorical]
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),numeric),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),categorical)])
models={'Logistic regression':LogisticRegression(max_iter=2000,class_weight='balanced'),'MLP':MLPClassifier(hidden_layer_sizes=(64,32),max_iter=500,random_state=SEED),'Random forest':RandomForestClassifier(n_estimators=500,class_weight='balanced',random_state=SEED,n_jobs=-1),'SVM':SVC(class_weight='balanced')}
if XGBClassifier is not None: models['XGBoost']=XGBClassifier(n_estimators=300,max_depth=3,learning_rate=.05,subsample=.8,colsample_bytree=.8,random_state=SEED,n_jobs=-1)
cv=StratifiedGroupKFold(5,shuffle=True,random_state=SEED); scoring={'accuracy':'accuracy','precision':'precision','recall':'recall','specificity':make_scorer(recall_score,pos_label=0)}
rows=[]
for name,estimator in models.items():
    result=cross_validate(Pipeline([('pre',pre),('model',estimator)]),X,y,groups=groups,cv=cv,scoring=scoring,n_jobs=1)
    rows.append({'model':name,**{metric+'_pct':100*np.mean(result['test_'+metric]) for metric in scoring}})
table=pd.DataFrame(rows).sort_values('accuracy_pct',ascending=False); display(table.round(2)); table.to_csv(RESULTS/'survey_only_model_comparison.csv',index=False)
